# Traduire le français et le ngiemboon (guembon)

Le **ngiemboon** (aussi écrit *ngienboum*, *guembon*) est une langue des Grassfields, parlée à l’ouest du Cameroun, notamment à **Batcham** (Bamboutos). L’orthographe du Nouveau Testament 2007 suit l’**Alphabet général des langues camerounaises** : ŋ, ɛ, ɔ, ʉ, ẅ, et des tons (̀ ́ ̌ ̂). On ne les « corrige » pas : ce sont des lettres, pas du bruit.

Les sources sont listées dans `data/README.md` et le mode d’emploi dans `README.md`. Le NT aligné reste le gros du corpus ; le voice-over santé (Timene 2024) ajoute du français **parlé** et du vocabulaire médical. Le précis Anderson (SIL) fixe l’orthographe AGLC. Ndiola 2011 sert à comprendre la syntaxe, pas à polluer l’orthographe d’entraînement.

Ancien tableau NT :

| Fichier | Langue | Couverture |
| --- | --- | --- |
| `Francais-Bible-Segond-1910-All-Bible.pdf` | français (Segond 1910) | Bible entière |
| `Ngiemboon-Bible-New-Testasment.pdf` | ngiemboon (Alliance biblique du Cameroun, 2007) | **Nouveau Testament seulement** |

On n’aligne donc **que le NT**. Un verset français et un verset ngiemboon qui portent la même référence (`Jean 3.16`, `Matthieu 1.1`, …) forment une paire d’apprentissage.

Un seul Transformer apprend **les deux sens**, comme dans le travail de Google sur la traduction multilingue : on préfixe la source par `<2ng>` (vers le ngiemboon) ou `<2fr>` (vers le français). Doubler le corpus sans doubler le modèle.

Ce n’est pas encore un produit de production. Huit mille versets, c’est précieux mais trop peu pour un grand réseau. Le modèle est **petit** exprès, le pipeline est **le même** que tu réutiliseras le jour où tu auras des sermons, des dialogues, des contes. Ajouter des paires, c’est ajouter des lignes dans `processed/pairs.jsonl` et relancer l’entraînement.


In [ ]:
from pathlib import Path
import sys

here = Path.cwd()
root = here if (here / "nmt").exists() else here / "Transformer"
sys.path.insert(0, str(root))
print("project root:", root)


## 1. Extraire et aligner les versets

Chaque page PDF est du texte (pas une image). On enlève en-têtes, numéros de page, références croisées du type `V. 1-17: cf. …` ou `(Lukase 3.23-38)`, puis on découpe :

1. un **livre + chapitre** (`Matthieu 3`, `Matio 3`, `Megàŋ ntúm 2`, `1 Korento 13`) ;
2. des **versets** : un nombre collé à une lettre, `16Car Dieu a tant aimé…` / `16Á nue ntí gie e Ssé…`.

La clé d’alignement est canonique : `JHN.3.16`. Si un verset n’existe que d’un côté, on le laisse de côté plutôt que d’inventer un parallèle.


In [ ]:
import json
from nmt.parse_bibles import PROCESSED_DIR, build_corpus

pairs = build_corpus(force=False)
stats = json.loads((PROCESSED_DIR / "stats.json").read_text(encoding="utf-8"))
print(json.dumps(stats, ensure_ascii=False, indent=2))
print()
for row in pairs:
    if row["id"] in {"MAT.1.1", "JHN.3.16"}:
        print(row["ref"])
        print(" FR:", row["fr"])
        print(" NG:", row["ng"])
        print()


## 2. Un vocabulaire commun (BPE)

Le français et le ngiemboon partagent des noms propres (Abraham, Jérusalem) mais pas la morphologie. Un **BPE joint** apprend des morceaux utiles aux deux côtés, **sans casser** les caractères AGLC : `ŋ`, `ẅ`, un ton, restent des symboles atomiques.

Jetons spéciaux :

| Jeton | Rôle |
| --- | --- |
| `<pad>` | remplissage d’un batch |
| `<bos>` / `<eos>` | début / fin de séquence |
| `<2ng>` | « traduis vers le ngiemboon » |
| `<2fr>` | « traduis vers le français » |


In [ ]:
from nmt.train import prepare

all_pairs, train_pairs, val_pairs, tokenizer = prepare(force_extract=False)
print("train verses", len(train_pairs), "val verses", len(val_pairs))
print("vocab", tokenizer.vocab_size)
print(tokenizer.encode("Car Dieu a tant aimé le monde", lang_tag="ng")[:20])
print(tokenizer.decode(tokenizer.encode("Car Dieu a tant aimé le monde", lang_tag="ng")))


## 3. Le modèle

Même architecture que `transformer.ipynb` : attention multi-têtes, FFN, encodage positionnel sinusoïdal, résiduelles, LayerNorm, masque causal côté décodeur. On réduit juste la taille (`d_model = 128`, 2 couches) pour que l’entraînement tienne sur CPU avec ~8 000 paires.

À l’apprentissage, chaque verset devient **deux** exemples :

```text
<2ng> + français     →  ngiemboon
<2fr> + ngiemboon    →  français
```

N’entraîne **pas** ici si tu n’as pas de GPU. Sur la machine d’un ami : `python -m nmt.corpus --force` puis `python -m nmt.train --device auto --no-resume`. La cellule suivante charge un checkpoint s’il existe, sinon elle s’arrête avec le message à lancer ailleurs.


In [ ]:
from nmt.paths import CHECKPOINTS
from nmt.train import load_model

ckpt = CHECKPOINTS / "fr_ngiemboon.pt"
if ckpt.exists():
    model, tokenizer, device = load_model()
    print("loaded", ckpt, "on", device)
else:
    print("Pas de checkpoint. Sur un GPU : python -m nmt.train --device auto --no-resume")


## 4. Traduire dans les deux sens

Décodage **glouton** : à chaque pas on prend le jeton le plus probable, jusqu’à `<eos>`. Ce n’est pas du beam search : plus tard, tu pourras l’ajouter dans `nmt/decode.py` sans toucher au reste.

Les versets du Nouveau Testament que le modèle a vus doivent déjà « ressembler » à quelque chose. Une phrase de tous les jours (marché, salutation) sera plus fragile : le corpus est biblique. C’est précisément ce que plus de données changeront.


In [ ]:
from nmt.decode import translate_pair

examples_fr = [
    "Généalogie de Jésus-Christ, fils de David, fils d’Abraham",
    "Car Dieu a tant aimé le monde qu’il a donné son Fils unique, afin que quiconque croit en lui ne périsse point, mais qu’il ait la vie éternelle",
    "Je suis le chemin, la vérité, et la vie",
]

for text in examples_fr:
    hyp = translate_pair(model, tokenizer, text, "fr2ng", device)
    print("FR:", text)
    print("NG:", hyp)
    print()


In [ ]:
examples_ng = [
    "Mezíŋ pàmangʉa njʉ̀’ Yěsô Kristo gẅie aa na ngẅɛ fùɔ Ndavíd, éfóo na ngẅɛ Ábalam e mɔɔn",
    "Á nue ntí gie e Ssé la lɔg ńkwoŋ njÿó pú’u la gwɔ́ tà á náa Míŋ we wémɔ’ɔ́, légwɔ́ gie wɛ̌ nyìŋ wɔ́ gẅie à zǎb ntʉ́m wé ná Míŋ we ŋwɛ́, à tě ńdá’a ńkaa ndòon tèmì wɔ́, à ge lá’ gẅiin le gwɔ́ tèmì",
]

for text in examples_ng:
    hyp = translate_pair(model, tokenizer, text, "ng2fr", device)
    print("NG:", text)
    print("FR:", hyp)
    print()


## 5. Tes propres phrases

Change `phrase` et `direction`, relance la cellule.


In [ ]:
phrase = "Que la grâce et la paix vous soient données"
direction = "fr2ng"  # ou "ng2fr"

print(translate_pair(model, tokenizer, phrase, direction, device))


## 6. Ligne de commande et suite

Depuis le dossier `Transformer`, une fois le checkpoint entraîné :

```bash
python -m nmt.translate --dir fr2ng "Je suis le chemin, la vérité, et la vie"
python -m nmt.translate --dir ng2fr "má"
```

Pour **réentraîner** après avoir ajouté des paires (même format JSONL : `id`, `fr`, `ng`) :

```bash
python -m nmt.train
```

Fichiers utiles :

| Chemin | Rôle |
| --- | --- |
| `nmt/parse_bibles.py` | PDF → versets alignés |
| `nmt/tokenizer.py` | BPE joint |
| `nmt/model.py` | Transformer |
| `nmt/train.py` | boucle d’apprentissage |
| `nmt/decode.py` / `nmt/translate.py` | inférence |
| `processed/pairs.jsonl` | corpus |
| `checkpoints/fr_ngiemboon.pt` | poids |

Le texte ngiemboon est © Alliance biblique du Cameroun, 2007. La Segond 1910 est dans le domaine public. Garde le NT pour l’apprentissage personnel ; ne republie pas le corpus tel quel.

Quand tu auras d’autres textes (prières, conversations, enseignement), ajoute-les comme de nouvelles paires. Le modèle n’a pas besoin de changer de forme : seulement de voir plus de langue de Batcham hors du registre biblique.

Interface : `python -m nmt.app` (n’entraîne pas). Entraînement GPU : `python -m nmt.train --device auto --no-resume`.
